In [17]:
import sqlite3
import pandas as pd
from loguru import logger

In [18]:
def is_sqlite(log_record):
    return log_record["extra"].get("sqlite") is True

logger.remove()
logger.add("DWH.log", level="DEBUG", filter=lambda r: not is_sqlite(r))
logger.add("DWH_SQlite.log", level="INFO", filter=is_sqlite)

sqliteLogger = logger.bind(sqlite= True)

In [19]:
conn_SDM = sqlite3.connect("../SDM/GO_SDM.db")
conn_DWH = sqlite3.connect("GO_DWH.db")
conn_DWH.set_trace_callback(sqliteLogger.info)

In [ ]:
RIM_SCRIPT = "DWH_RIM.txt"

def create_dwh_db():
    conn_DWH.set_trace_callback()

    with open(RIM_SCRIPT, "r") as f:
        sql = f.read()

    try:
        conn_DWH.executescript(sql)
        conn_DWH.commit()
        logger.info("DWH database aangemaakt op basis van RIM script")
    except Exception as e:
        logger.error(f"fout bij aanmaken DWH database: {e}")
        raise


    conn_DWH.set_trace_callback(sqliteLogger.info)

#create_dwh_db()

In [ ]:
# COUNTRY - Not directly in DWH
# Cached class
class country:
    countries = None

    def get(self):
        # Cached getter that returns a copy
        if self.countries is None:
            self._create()
        return self.countries.copy()

    def clear(self):
        self.countries = None

    def _create(self):
        # Get SDM Values
        SDM_crm_country = pd.read_sql_query("SELECT * FROM Crm_Country", conn_SDM)
        SDM_sales_country = pd.read_sql_query("SELECT * FROM country", conn_SDM)
        SDM_sales_territory = pd.read_sql_query("SELECT * FROM Sales_Territory", conn_SDM)

        # JOIN sales_territory + crm_country
        SDM_crm_country = ((SDM_crm_country.merge(SDM_sales_territory, how="left")
                           .drop(columns={'SALES_TERRITORY_CODE'}))
                           .rename(columns={"COUNTRY_EN":"COUNTRY"}))
        # Multiload crm_country + sales_country
        self.countries = SDM_crm_country.merge(SDM_sales_country, how="outer")

def countries():
    # Get SDM Values
    SDM_crm_country = pd.read_sql_query("SELECT * FROM Crm_Country", conn_SDM)
    SDM_sales_country = pd.read_sql_query("SELECT * FROM country", conn_SDM)
    SDM_sales_territory = pd.read_sql_query("SELECT * FROM Sales_Territory", conn_SDM)

    # JOIN sales_territory + crm_country
    SDM_crm_country = ((SDM_crm_country.merge(SDM_sales_territory, how="left")
                       .drop(columns={'SALES_TERRITORY_CODE'}))
                       .rename(columns={"COUNTRY_EN":"COUNTRY"}))
    # Multiload crm_country + sales_country
    return SDM_crm_country.merge(SDM_sales_country, how="outer")

In [47]:
# Customer Site - Type 2
# Merge HQ + Country
SDM_Customer_Headquarters = pd.read_sql_query("SELECT * FROM Customer_Headquarters", conn_SDM)
SDM_Customer_Headquarters = SDM_Customer_Headquarters.merge(countries(), how="left").drop(columns={"COUNTRY_CODE"})
SDM_Customer_Headquarters.columns = "HQ_" + SDM_Customer_Headquarters.columns.values
SDM_Customer_Headquarters = SDM_Customer_Headquarters.rename(columns={"HQ_CUSTOMER_CODEMR": "CUSTOMER_CODEMR", "HQ_SEGMENT_CODE": "SEGMENT_CODE"})

# Merge HQ + Customer Segment
SDM_Customer_Segment = pd.read_sql_query("SELECT * FROM Customer_Segment", conn_SDM)
SDM_Customer_Headquarters = SDM_Customer_Headquarters.merge(SDM_Customer_Segment, how="left").drop(columns={"SEGMENT_CODE"})

# Merge Age Group + Sales Demographic
SDM_Age_Group = pd.read_sql_query("SELECT * FROM Age_Group", conn_SDM)
SDM_Sales_Demographic = pd.read_sql_query("SELECT * FROM Sales_Demographic", conn_SDM)
SDM_Sales_Demographic = SDM_Sales_Demographic.merge(SDM_Age_Group, how="left").drop(columns={"AGE_GROUP_CODE"})

# Transform SALES_PERCENT to age brackets
for lower_age in SDM_Sales_Demographic["LOWER_AGE"].unique():
    for upper_age in SDM_Sales_Demographic.loc[SDM_Sales_Demographic["LOWER_AGE"] == lower_age, "UPPER_AGE"].unique():
        SDM_Sales_Demographic.loc[(SDM_Sales_Demographic["LOWER_AGE"] == lower_age) & (SDM_Sales_Demographic["UPPER_AGE"] == upper_age), f"AGE{lower_age}_{upper_age}"] = SDM_Sales_Demographic["SALES_PERCENT"]

# Fold age brackets in to a single demographics row
SDM_Sales_Demographic = SDM_Sales_Demographic.drop(columns={"DEMOGRAPHIC_CODE", "SALES_PERCENT", "UPPER_AGE", "LOWER_AGE"})
SDM_Sales_Demographic = SDM_Sales_Demographic.set_index("CUSTOMER_CODEMR").groupby(level=0).transform(lambda x: sorted(x, key=lambda k: pd.isna(k))).dropna(how="all")

# Merge HQ + Sales Demographic
SDM_Customer_Headquarters = SDM_Customer_Headquarters.merge(SDM_Sales_Demographic, how="left", on="CUSTOMER_CODEMR")

# Merge Customer Type + Customer
SDM_Customer_Type = pd.read_sql_query("SELECT * FROM Customer_Type", conn_SDM)
SDM_Customer = pd.read_sql_query("SELECT * FROM Customer", conn_SDM)

# Merge HQ + Customer
SDM_Customer = SDM_Customer.merge(SDM_Customer_Type, how="left").drop(columns={"CUSTOMER_TYPE_CODE"})
SDM_Customer = SDM_Customer.merge(SDM_Customer_Headquarters, how="outer")
SDM_Customer["COMPANY_NAME"] = SDM_Customer["COMPANY_NAME"].fillna(SDM_Customer["HQ_CUSTOMER_NAME"])
SDM_Customer = SDM_Customer.drop(columns={"HQ_CUSTOMER_NAME", "CUSTOMER_CODEMR"})

# Get Customer Store + Retailer Site
SDM_Customer_Store = pd.read_sql_query("SELECT CUSTOMER_CODE, STREET as ADDRESS1, ADDITION as ADDRESS2, CITY, STATE as REGION, ZIPCODE as POSTAL_ZONE, COUNTRY_CODE, ACTIVE_INDICATOR, CUSTOMER_SITE_CODE as PK_CUSTOMER_STORE FROM Customer_Store", conn_SDM)
SDM_Retailer_Site = pd.read_sql_query("SELECT RETAILER_CODE as CUSTOMER_CODE, ADDRESS1, ADDRESS2, CITY, REGION, POSTAL_ZONE, COUNTRY_CODE, ACTIVE_INDICATOR, RETAILER_SITE_CODE as PK_RETAILER_SITE FROM retailer_site", conn_SDM)

# Multiload Customer Store + Retailer Site, merge with country, merge with customer
SDM_Customer_Site = SDM_Customer_Store.merge(SDM_Retailer_Site, how="outer")
SDM_Customer_Site = SDM_Customer_Site.merge(countries(), how="left").drop(columns={"COUNTRY_CODE"})
SDM_Customer_Site = SDM_Customer_Site.merge(SDM_Customer, how="outer").drop(columns={"CUSTOMER_CODE"})

# Get and prepare DWH values
DWH_Customer_Site = pd.read_sql_query("SELECT * FROM customer_site WHERE VALID_TILL IS NULL", conn_DWH)
DWH_Customer_Site = DWH_Customer_Site.drop(columns={"VALID_FROM", "VALID_TILL"})
DWH_Customer_Site = DWH_Customer_Site.astype({"ACTIVE_INDICATOR": "float64", "PK_CUSTOMER_STORE": "float64", "PK_RETAILER_SITE": "float64"})

# Determine difference between SDM and DWH
diff = SDM_Customer_Site.merge(DWH_Customer_Site, indicator=True, how="outer")
new = diff.loc[(diff["_merge"] == "left_only")].drop("_merge", axis=1)
outdated = diff.loc[(diff["_merge"] == "right_only"), ["CUSTOMER_STORE_SK", "PK_CUSTOMER_STORE", "PK_RETAILER_SITE"]]

# Determine if values are new or updated
new = (new.drop(['CUSTOMER_STORE_SK'], axis=1)
       .merge(outdated.loc[outdated['PK_CUSTOMER_STORE'].notna(), ["CUSTOMER_STORE_SK", "PK_CUSTOMER_STORE"]], left_on="PK_CUSTOMER_STORE", right_on="PK_CUSTOMER_STORE", how="left")
       .merge(outdated.loc[outdated['PK_RETAILER_SITE'].notna(), ["CUSTOMER_STORE_SK", "PK_RETAILER_SITE"]], left_on="PK_RETAILER_SITE", right_on="PK_RETAILER_SITE", how="left"))
new["already_exists"] = new["CUSTOMER_STORE_SK_x"].notna() | new["CUSTOMER_STORE_SK_y"].notna()

# Update DWH
cur = conn_DWH.cursor()
try:
    # Set valid till date for outdated DWH values
    for record in outdated.to_dict(orient="records"):
        try:
            cur.execute("UPDATE customer_site SET valid_till = DATE('now') WHERE CUSTOMER_STORE_SK = :CUSTOMER_STORE_SK", record)
        except sqlite3.Error as er:
            sqliteLogger.error(er)
            logger.info("Fout bij zetten van einddatum in database: {database} PK:{key}", database="customer_site", key=record["CUSTOMER_STORE_SK"])

    # Insert new values in to DWH
    for record in new.to_dict(orient="records"):
        try:
            cur.execute("INSERT INTO "
                        "customer_site(ADDRESS1, ADDRESS2, CITY, REGION, POSTAL_ZONE, TERRITORY_NAME_EN, FLAG_IMAGE, CURRENCY_NAME, LANGUAGE, COUNTRY, COMPANY_NAME, CUSTOMER_TYPE_EN, SEGMENT_DESCRIPTION, SEGMENT_NAME, HQ_ADDRESS1, HQ_ADDRESS2, HQ_CITY, HQ_REGION, HQ_POSTAL_ZONE, HQ_PHONE, HQ_FAX, HQ_LANGUAGE, HQ_CURRENCY_NAME, HQ_FLAG_IMAGE, HQ_TERRITORY_NAME_EN, HQ_COUNTRY, AGE61_70, AGE51_60, AGE41_50, AGE31_40, AGE21_30, AGE0_20, ACTIVE_INDICATOR, PK_CUSTOMER_STORE, PK_RETAILER_SITE, VALID_FROM) VALUES(:ADDRESS1, :ADDRESS2, :CITY, :REGION, :POSTAL_ZONE, :TERRITORY_NAME_EN, :FLAG_IMAGE, :CURRENCY_NAME, :LANGUAGE, :COUNTRY, :COMPANY_NAME, :CUSTOMER_TYPE_EN, :SEGMENT_DESCRIPTION, :SEGMENT_NAME, :HQ_ADDRESS1, :HQ_ADDRESS2, :HQ_CITY, :HQ_REGION, :HQ_POSTAL_ZONE, :HQ_PHONE, :HQ_FAX, :HQ_LANGUAGE, :HQ_CURRENCY_NAME, :HQ_FLAG_IMAGE, :HQ_TERRITORY_NAME_EN, :HQ_COUNTRY, :AGE61_70, :AGE51_60, :AGE41_50, :AGE31_40, :AGE21_30, :AGE0_20, :ACTIVE_INDICATOR, :PK_CUSTOMER_STORE, :PK_RETAILER_SITE, (CASE WHEN :already_exists THEN DATE('now') ELSE DATE('1970-01-01') END))", record)
        except sqlite3.Error as er:
            sqliteLogger.error(er)
            logger.info("Fout bij inladen in database: {database} PK:{key}", database="customer_site", key=record["CUSTOMER_STORE_SK"])

finally:
    cur.close()
conn_DWH.commit()
logger.info("tabel {tabel} geüpdated, {new} rijen toegevoegd, {outdated} rijen voorzien van einddatum", tabel="customer_site", new=len(new.index), outdated=len(outdated.index))


In [34]:
# Customer Contact - Type 2
# Get SDM values
SDM_Customer_Contact = pd.read_sql_query("SELECT * FROM Customer_Contact", conn_SDM)
SDM_Customer_Contact = SDM_Customer_Contact.astype({"EXTENSION": "float64"})

# Determine name
SDM_Customer_Contact["NAME"] = SDM_Customer_Contact["FIRST_NAME"] + " " + SDM_Customer_Contact["LAST_NAME"]

# Get current DWH SK-PK connections and apply
DWH_Customer_Site = pd.read_sql_query("SELECT CUSTOMER_STORE_SK, PK_RETAILER_SITE FROM customer_site WHERE VALID_TILL IS NULL", conn_DWH)
SDM_Customer_Contact = SDM_Customer_Contact.merge(DWH_Customer_Site, how="left", left_on="CUSTOMER_SITE_CODE", right_on="PK_RETAILER_SITE").drop(columns={"CUSTOMER_SITE_CODE", "PK_RETAILER_SITE"})

# Get and prepare DWH values
DWH_Customer_Contact = pd.read_sql_query("SELECT * FROM customer_contact WHERE VALID_TILL IS NULL", conn_DWH)
DWH_Customer_Contact = DWH_Customer_Contact.drop(columns={"VALID_FROM", "VALID_TILL"})

# Determine difference between SDM and DWH
diff = SDM_Customer_Contact.merge(DWH_Customer_Contact, indicator=True, how="outer")
new = diff.loc[(diff["_merge"] == "left_only")].drop("_merge", axis=1)
outdated = diff.loc[(diff["_merge"] == "right_only"), ["CUSTOMER_CONTACT_SK", "CUSTOMER_CONTACT_CODE"]]

# Determine if values are new or updated
new = new.drop(['CUSTOMER_CONTACT_SK'], axis=1).merge(outdated, how="left")
new["already_exists"] = new["CUSTOMER_CONTACT_SK"].notna()

# Update DWH
cur = conn_DWH.cursor()
try:
    # Set valid till date for outdated DWH values
    for record in outdated.to_dict(orient="records"):
        try:
            cur.execute("UPDATE customer_contact SET valid_till = DATE('now') WHERE CUSTOMER_CONTACT_SK = :CUSTOMER_CONTACT_SK", record)
        except sqlite3.Error as er:
            sqliteLogger.error(er)
            logger.info("Fout bij zetten van einddatum in database: {database} PK:{key}", database="customer_contact", key=record["CUSTOMER_CONTACT_SK"])

    # Insert new values in to DWH
    for record in new.to_dict(orient="records"):
        try:
            cur.execute("INSERT INTO "
                        "customer_contact(CUSTOMER_STORE_SK, NAME, JOB_POSITION_EN, EXTENSION, GENDER, CUSTOMER_CONTACT_CODE, VALID_FROM) VALUES(:CUSTOMER_STORE_SK, :NAME, :JOB_POSITION_EN, :EXTENSION, :GENDER, :CUSTOMER_CONTACT_CODE, (CASE WHEN :already_exists THEN DATE('now') ELSE DATE('1970-01-01') END))", record)
        except sqlite3.Error as er:
            sqliteLogger.error(er)
            logger.info("Fout bij inladen in database: {database} PK:{key}", database="customer_contact", key=record["CUSTOMER_CONTACT_SK"])

finally:
    cur.close()
conn_DWH.commit()
logger.info("tabel {tabel} geüpdated, {new} rijen toegevoegd, {outdated} rijen voorzien van einddatum", tabel="customer_contact", new=len(new.index), outdated=len(outdated.index))


In [ ]:
# Order Method - Type 1
# Get SDM values
SDM_Order_Method = pd.read_sql_query("SELECT * FROM order_method", conn_SDM)

# Get DWH values
DWH_Order_Method = pd.read_sql_query("SELECT * FROM order_method", conn_DWH)

# Determine difference between SDM and DWH
diff = SDM_Order_Method.merge(DWH_Order_Method, indicator=True, how='outer')
new = diff.loc[(diff["_merge"] == "left_only")]
removed = diff.loc[(diff["_merge"] == "right_only")]

# Update DWH
cur = conn_DWH.cursor()
try:
    # Delete removed and updated values from DWH
    for record in removed.to_dict(orient="records"):
        try:
            cur.execute(f"DELETE FROM Order_Method WHERE ORDER_METHOD_CODE = :ORDER_METHOD_CODE", record)
        except sqlite3.Error as er:
            sqliteLogger.error(er)
            logger.info("Fout bij verwijderen uit database: {database} PK:{key}", database="order_method", key=record["ORDER_METHOD_CODE"] )

    # Insert new values
    for record in new.to_dict(orient="records"):
        try:
            cur.execute(f"INSERT INTO ORDER_METHOD VALUES (:ORDER_METHOD_CODE, :ORDER_METHOD_EN)", record)
        except sqlite3.Error as er:
            sqliteLogger.error(er)
            logger.info("Fout bij inladen in database: {database} PK:{key}", database="order_method", key=record["ORDER_METHOD_CODE"] )
finally:
    cur.close()
conn_DWH.commit()
logger.info("tabel {tabel} geüpdated, {new} rijen toegevoegd, {removed} rijen verwijderd", tabel="order_method", new=len(new.index), removed=len(removed.index))

In [1]:
# Return Reason - Type 1

In [2]:
# Course - Type 1

In [3]:
# Satisfaction Type - Type 1

In [4]:
# Sales Branch - Type 2

In [5]:
# Sales Representative - Type 2

In [6]:
# Product - Type 1

In [7]:
# Date - Type 1

In [8]:
# Month Year - Type 1

In [9]:
# Year - Type 1

In [10]:
# Order - Type 1

In [11]:
# Return - Type 1

In [12]:
# Training - Type 1

In [13]:
# Satisfaction - Type 1

In [14]:
# Product Forecast - Type 1

In [15]:
# Inventory Level - Type 1

In [ ]:
# Sales Target - Type 1